In [1]:
from dotenv import load_dotenv
from src.models.openai_provider import OpenAILLMProvider
from src.prompts import language_detection_system, language_detection_user
from src.models.schemas import LanguageDetection
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.db.sq_lite import SqLite


load_dotenv()

/home/lamossta/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Downloading and creating SPARQL endpoint for the NKOD metadata

In [2]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)

#nkod_data_processor.download_catalog_metadata()
#nkod_data_processor.create_metadata_csv(graph_db)
nkod_data_processor.create_metadata_sql(sq_lite)

## Indexing the keywords, titles and descriptions from the NKOD metadata (TODO: VectorDb Chroma)

In [3]:
nkod_data_processor.index_catalog_metadata(sq_lite)

Mean number of cs keywords: 3.7603933839964236
Mean number of en keywords: 3.0932371160355068
Mean number of cs descriptions that are not None: 0.9977010026183025
Mean number of en descriptions that are not None: 0.8124720607957086
Mean number of cs titles that are not None: 1.0
Mean number of en titles that are not None: 0.8264576282010345


## Language detection of the input query

In [4]:
input_query = "datasety o duchodech"

model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)
response = openai_llm.chat(
    user_prompt=language_detection_user[model_name],
    user_prompt_vars={
        "text": input_query
    },
    system_prompt = language_detection_system[model_name],
    structured_output=LanguageDetection
)

print(response)

czech


## Timeframe detection of the input query